# Preprocesamiento PPG — Módulo C1 (BIDMC)

**Objetivo:** pipeline ETL de la señal PPG de la capa `bronce`, con filtrado, ventaneo, índices de calidad (SQI), validación de frecuencia cardíaca (FC) y export a la capa `oro` a 125 Hz.

**Este notebook corresponde a C1:** procesamiento y validación sobre BIDMC. **No entrena modelo de SpO2** (C2 bloqueado: BIDMC no tiene canales Rojo/IR ni ground truth independiente; ver plan).

### Capas de datos
* **`bronce/ppg`**: `BIDMC-Reduced.csv` (inmutable).
* **`plata/ppg`**: auditoría física, SQI por ventana, validación de FC y figuras.
* **`oro/ppg`**: `set_a.parquet` (BIDMC a 125 Hz).

### Límites de BIDMC
* SpO2 en normoxia (83–100 %): no permite evaluar desaturaciones.
* `PLETH` monocanal procesado por el monitor (sin Rojo/IR).
* Registros de UCI inmóvil, sin acelerómetro (no se valida SQI de movimiento).

### Referencias
* Pimentel et al. (2017), *IEEE TBME*, 64(8), 1914–1923 — BIDMC. https://doi.org/10.1109/TBME.2016.2613124
* Makowski et al. (2021), *Behavior Research Methods*, 53(4), 1689–1696 — NeuroKit2. https://doi.org/10.3758/s13428-020-01516-y
* Elgendi et al. (2013), *PLoS ONE*, 8(10), e76585 — detección de picos sistólicos. https://doi.org/10.1371/journal.pone.0076585

### Enlaces
* [Ficha del dataset BIDMC](../../../doc/03_DATASETS/Elegidos/BIDMC.md)
* [Plan de Implementación del Módulo C](../../../doc/00_PLAN/Plan_Implementacion_Módulo_C.md)

In [ ]:
import json
from datetime import datetime, timezone
import pathlib
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import welch
from scipy.stats import skew, kurtosis, pearsonr
import neurokit2 as nk

%matplotlib inline

# --- Capas de datos ---------------------------------------------------------
BRONZE_DIR = pathlib.Path("../../data/bronce/ppg")
PLATA_DIR = pathlib.Path("../../data/plata/ppg")
ORO_DIR = pathlib.Path("../../data/oro/ppg")

# --- Parámetros del pipeline ------------------------------------------------
FS = 125             # Frecuencia de muestreo nativa de BIDMC (Hz)
WINDOW_S = 8         # Ventana de análisis (s)
STEP_S = 4           # Paso entre ventanas (s), solape 50 %
BANDPASS = (0.5, 8)  # Banda de pulso de nk.ppg_clean (Elgendi et al., 2013)
SEED = 42

# --- Rangos fisiológicos de auditoría ---------------------------------------
RANGE_SPO2 = (70, 100)  # %
RANGE_HR = (30, 220)    # bpm
RANGE_RR = (4, 60)      # resp/min

WINDOW_N = int(WINDOW_S * FS)  # 1000 muestras
STEP_N = int(STEP_S * FS)      # 500 muestras
EPS = np.finfo(float).eps

np.random.seed(SEED)

def save_json_local(data: dict, name: str) -> int:
    PLATA_DIR.mkdir(parents=True, exist_ok=True)
    path = PLATA_DIR / name
    path.write_text(json.dumps(data, indent=2, default=str), encoding="utf-8")
    return path.stat().st_size

def save_csv_local(data: pd.DataFrame, name: str) -> int:
    PLATA_DIR.mkdir(parents=True, exist_ok=True)
    path = PLATA_DIR / name
    data.to_csv(path, index=False)
    return path.stat().st_size

print(f"NeuroKit2 {nk.__version__} | FS={FS} Hz | ventana={WINDOW_S} s ({WINDOW_N} muestras) | paso={STEP_S} s")

In [ ]:
df = pd.read_csv(BRONZE_DIR / "BIDMC-Reduced.csv")

EXPECTED_COLS = ["Record", "Subject", "Sample_Index", "Time_s", "PLETH", "HR", "PULSE", "RR", "SpO2"]
EXPECTED_ROWS = 3_180_053
EXPECTED_RECORDS = 53
EXPECTED_SUBJECTS = 46

assert list(df.columns) == EXPECTED_COLS, f"Esquema inesperado: {list(df.columns)}"
assert len(df) == EXPECTED_ROWS, f"Filas: {len(df):,} != {EXPECTED_ROWS:,}"
assert df["Record"].nunique() == EXPECTED_RECORDS, f"Records: {df['Record'].nunique()}"
assert df["Subject"].nunique() == EXPECTED_SUBJECTS, f"Sujetos: {df['Subject'].nunique()}"

nan_raw = df.isna().sum()
assert int(nan_raw.sum()) == 0, f"NaN crudos inesperados:\n{nan_raw[nan_raw > 0]}"

df["PLETH"] = df["PLETH"].astype(np.float32)
assert df["PLETH"].dtype == np.float32

records_list = sorted(df["Record"].unique())

print(f"BIDMC cargado: {len(df):,} filas | {len(records_list)} records | {df['Subject'].nunique()} sujetos")
print(f"NaN crudos: {int(nan_raw.sum())} | PLETH: {df['PLETH'].dtype}")
print(f"Muestras por registro: {df.groupby('Record').size().unique()} | Time_s: {df['Time_s'].min():.3f}–{df['Time_s'].max():.3f} s")

In [ ]:
audit = {}
for rec, g in df.groupby("Record", sort=True):
    pleth = g["PLETH"].to_numpy(dtype=np.float64)
    spo2 = g["SpO2"].to_numpy(dtype=np.float64)
    hr = g["HR"].to_numpy(dtype=np.float64)
    rr = g["RR"].to_numpy(dtype=np.float64)

    std_pleth = float(np.std(pleth))
    subject = g["Subject"].iloc[0]
    audit[rec] = {
        "Subject": int(subject) if isinstance(subject, np.integer) else subject,
        "N_Samples": int(len(g)),
        "NaN": int(np.isnan(pleth).sum() + np.isnan(spo2).sum() + np.isnan(hr).sum() + np.isnan(rr).sum()),
        "Flatline": bool(std_pleth < 1e-9),
        "Pct_SpO2_Out": float(((spo2 < RANGE_SPO2[0]) | (spo2 > RANGE_SPO2[1])).mean() * 100),
        "Pct_HR_Out": float(((hr < RANGE_HR[0]) | (hr > RANGE_HR[1])).mean() * 100),
        "Pct_RR_Out": float(((rr < RANGE_RR[0]) | (rr > RANGE_RR[1])).mean() * 100),
        "AC_DC_pct": float((pleth.max() - pleth.min()) / abs(pleth.mean()) * 100),
        "PLETH_Mean": float(pleth.mean()),
        "PLETH_Std": std_pleth,
    }

df_audit = pd.DataFrame(audit).T
print("Auditoría física por registro (primeras 10 filas):")
print(df_audit[["Subject", "N_Samples", "NaN", "Flatline", "Pct_SpO2_Out", "Pct_HR_Out", "Pct_RR_Out", "AC_DC_pct"]].round(3).head(10).to_string())

audit_doc = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "dataset": "BIDMC-Reduced.csv",
    "fs": FS,
    "ranges": {"SpO2": list(RANGE_SPO2), "HR": list(RANGE_HR), "RR": list(RANGE_RR)},
    "n_records": len(audit),
    "records": audit,
}
audit_bytes = save_json_local(audit_doc, "bidmc_audit.json")

print(f"\nResumen: {len(audit)} registros | NaN={int(df_audit['NaN'].astype(int).sum())} | flatline={int(df_audit['Flatline'].astype(bool).sum())}")
print(f"Media % fuera de rango SpO2/HR/RR = {df_audit['Pct_SpO2_Out'].mean():.3f}/{df_audit['Pct_HR_Out'].mean():.3f}/{df_audit['Pct_RR_Out'].mean():.3f}")
print(f"AC/DC por registro: mediana {df_audit['AC_DC_pct'].median():.2f} % | rango {df_audit['AC_DC_pct'].min():.2f}–{df_audit['AC_DC_pct'].max():.2f} %")
print(f"bidmc_audit.json → {audit_bytes:,} bytes")

In [ ]:
proc: dict[str, dict] = {}
for i, (rec, g) in enumerate(df.groupby("Record", sort=True), 1):
    raw = g["PLETH"].to_numpy(dtype=np.float64)
    clean = nk.ppg_clean(raw, sampling_rate=FS, method="elgendi")
    assert len(clean) == len(raw), f"{rec}: longitudes distintas ({len(clean)} != {len(raw)})"

    proc[rec] = {
        "Subject": g["Subject"].iloc[0],
        "Raw": raw.astype(np.float32),
        "Clean": np.asarray(clean, dtype=np.float32),
        "SpO2": g["SpO2"].to_numpy(dtype=np.float32),
        "HR": g["HR"].to_numpy(dtype=np.float32),
        "RR": g["RR"].to_numpy(dtype=np.float32),
    }
    if i % 10 == 0 or i == len(records_list):
        print(f"  Filtrado {i}/{len(records_list)} registros")

print(f"Capa intermedia lista: {len(proc)} registros con (Raw, Clean, numéricos) a {FS} Hz")

In [ ]:
starts_by_record = {
    rec: np.arange(0, len(d["Clean"]) - WINDOW_N + 1, STEP_N)
    for rec, d in proc.items()
}
n_windows_total = int(sum(len(s) for s in starts_by_record.values()))

assert all(len(s) > 0 for s in starts_by_record.values()), "Algún registro quedó sin ventanas"

print(f"Ventaneo: ventana {WINDOW_S} s ({WINDOW_N} muestras), paso {STEP_S} s ({STEP_N} muestras)")
print(f"Ventanas por registro: {sorted({len(s) for s in starts_by_record.values()})} | Total: {n_windows_total:,}")

# Cobertura de numéricos: en BIDMC los numéricos llegan en step-hold con faltantes
# imputados en la reducción (D5/D10 de BIDMC.md), por lo que la cobertura por
# ventana es 100 % por construcción. El gate de cobertura SpO2 ≥90 % se reserva
# para los datasets de set_b (C2).

In [ ]:
peaks_by_record = {}

def window_metrics(raw_win: np.ndarray, clean_win: np.ndarray) -> tuple:
    # Métricas propias de ventana: SNR en banda de pulso, skewness, kurtosis y AC/DC
    freqs, pxx = welch(raw_win, fs=FS)
    in_band = (freqs >= BANDPASS[0]) & (freqs <= BANDPASS[1])
    p_band = float(pxx[in_band].sum())
    p_total = float(pxx.sum())
    ratio = p_band / max(p_total - p_band, EPS)
    snr_db = 10 * np.log10(max(ratio, EPS))

    std_raw = float(np.std(raw_win))
    dropout = bool(np.isnan(raw_win).any() or std_raw == 0)
    flatline = bool(std_raw < 1e-9)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        sk = float(skew(clean_win))
        ku = float(kurtosis(clean_win))
    ac_dc = float((clean_win.max() - clean_win.min()) / abs(raw_win.mean()) * 100)

    return snr_db, sk, ku, ac_dc, dropout, flatline

window_rows = []
for i, rec in enumerate(records_list, 1):
    d = proc[rec]
    clean = d["Clean"]
    raw = d["Raw"]

    _, info = nk.ppg_peaks(clean, sampling_rate=FS, method="elgendi")
    peaks = np.asarray(info["PPG_Peaks"])
    peaks_by_record[rec] = peaks

    if len(peaks) >= 2:
        quality = nk.ppg_quality(clean, peaks=peaks, sampling_rate=FS, method="templatematch")
        quality_pulse = np.asarray(quality)[peaks]
    else:
        quality_pulse = np.full(len(peaks), np.nan, dtype=np.float64)
    rate = nk.ppg_rate(peaks, sampling_rate=FS, desired_length=len(clean))

    for w_idx, start in enumerate(starts_by_record[rec]):
        end = start + WINDOW_N
        raw_win = raw[start:end]
        clean_win = clean[start:end]

        spo2_ref = float(np.mean(d["SpO2"][start:end]))
        hr_ref = float(np.mean(d["HR"][start:end]))
        rr_ref = float(np.mean(d["RR"][start:end]))

        # nk.ppg_quality devuelve un vector por muestra: _quality_templatematch
        # interpola la CC por ciclo a len(signal) (NeuroKit2 0.2.13). Se indexa en
        # los picos para obtener el SQI por pulso y se promedia por ventana con los
        # pulsos cuyo pico cae dentro de ella.
        in_win = (peaks >= start) & (peaks < end)
        n_peaks = int(in_win.sum())
        if n_peaks >= 2:
            hr_est = float(np.mean(rate[start:end]))
            sqi_values = quality_pulse[in_win]
            sqi = float(np.nanmean(sqi_values)) if not np.isnan(sqi_values).all() else np.nan
            hr_err = hr_est - hr_ref
        else:
            hr_est = np.nan
            hr_err = np.nan
            sqi = np.nan

        snr_db, sk, ku, ac_dc, dropout, flatline = window_metrics(raw_win, clean_win)

        window_rows.append({
            "Record": rec,
            "Subject": d["Subject"],
            "Window_Index": w_idx,
            "T_start_s": start / FS,
            "SpO2_Ref": spo2_ref,
            "HR_Ref": hr_ref,
            "RR_Ref": rr_ref,
            "HR_est": hr_est,
            "HR_err": hr_err,
            "SQI_templatematch": sqi,
            "SNR_dB": snr_db,
            "Skew": sk,
            "Kurt": ku,
            "AC_DC_pct": ac_dc,
            "N_Peaks": n_peaks,
            "Dropout": dropout,
            "Flatline": flatline,
        })

    if i % 10 == 0 or i == len(records_list):
        print(f"  Ventaneo {i}/{len(records_list)} registros")

dfw = pd.DataFrame(window_rows)
print(f"\nVentanas: {len(dfw):,} | registros: {dfw['Record'].nunique()} | ventanas con N_Peaks<2: {int((dfw['N_Peaks'] < 2).sum()):,}")
print(f"SQI por pulso disponible en {int(dfw['SQI_templatematch'].notna().sum()):,} ventanas")

In [ ]:
THR_SQI = float(np.nanpercentile(dfw["SQI_templatematch"], 10))
THR_SNR = float(np.nanpercentile(dfw["SNR_dB"], 10))
THR_ACDC = float(np.nanpercentile(dfw["AC_DC_pct"], 10))

# Un SQI NaN con N_Peaks >= 2 también falla el criterio por percentil
fail_sqi = (dfw["SQI_templatematch"] < THR_SQI) | ((dfw["N_Peaks"] >= 2) & dfw["SQI_templatematch"].isna())
fail_snr = dfw["SNR_dB"] < THR_SNR
fail_acdc = dfw["AC_DC_pct"] < THR_ACDC
fail_hard = (dfw["N_Peaks"] < 2) | dfw["Dropout"] | dfw["Flatline"]
dfw["N_Failures"] = fail_sqi.astype(int) + fail_snr.astype(int) + fail_acdc.astype(int)
dfw["Quality_Flag"] = np.where(fail_hard | (dfw["N_Failures"] >= 2), "low-quality", "ok")

print(f"Umbrales P10: SQI={THR_SQI:.4f} | SNR={THR_SNR:.2f} dB | AC/DC={THR_ACDC:.2f} %")
print(f"Quality_Flag: {dfw['Quality_Flag'].value_counts().to_dict()}")
print(f"Fallos: SQI={int(fail_sqi.sum())} | SNR={int(fail_snr.sum())} | AC/DC={int(fail_acdc.sum())} | duro (N_Peaks<2/dropout/flatline)={int(fail_hard.sum())}")

quality_config = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "fs": FS,
    "window_s": WINDOW_S,
    "step_s": STEP_S,
    "thresholds": {"method": "percentil 10", "SQI_templatematch": THR_SQI, "SNR_dB": THR_SNR, "AC_DC_pct": THR_ACDC},
    "flags": {
        "policy": "Quality_Flag = 'low-quality' si hay flag duro o si fallan >= 2 de los 3 criterios por percentil",
        "percentile_criteria": ["SQI_templatematch < umbral (o NaN con N_Peaks >= 2)", "SNR_dB < umbral", "AC_DC_pct < umbral"],
        "hard": "N_Peaks < 2 o Dropout o Flatline (fuerza 'low-quality')",
        "min_percentile_failures_low_quality": 2,
    },
    "counts": {
        "windows_total": int(len(dfw)),
        "ok": int((dfw["Quality_Flag"] == "ok").sum()),
        "low_quality": int((dfw["Quality_Flag"] == "low-quality").sum()),
        "fail_SQI": int(fail_sqi.sum()),
        "fail_SNR": int(fail_snr.sum()),
        "fail_AC_DC": int(fail_acdc.sum()),
        "fail_hard": int(fail_hard.sum()),
    },
    "by_record": {
        rec: {"ok": int((g["Quality_Flag"] == "ok").sum()), "low_quality": int((g["Quality_Flag"] == "low-quality").sum())}
        for rec, g in dfw.groupby("Record", sort=True)
    },
}
qc_bytes = save_json_local(quality_config, "ppg_quality_config.json")
print(f"\nppg_quality_config.json → {qc_bytes:,} bytes")

def hr_validation_summary(d: pd.DataFrame) -> dict:
    # MAE/RMSE/Pearson/Bland-Altman de HR_est vs HR_Ref en ventanas válidas
    valid = d[["HR_est", "HR_Ref"]].dropna()
    n_valid = int(len(valid))
    if n_valid < 2:
        return {"N_Windows": int(len(d)), "N_Valid": n_valid, "MAE": None, "RMSE": None,
                "Pearson_r": None, "Bias": None, "LoA_Lower": None, "LoA_Upper": None}
    if valid["HR_est"].std() == 0 or valid["HR_Ref"].std() == 0:
        r = None
    else:
        r = float(pearsonr(valid["HR_est"], valid["HR_Ref"])[0])
    err = valid["HR_est"] - valid["HR_Ref"]
    return {
        "N_Windows": int(len(d)),
        "N_Valid": n_valid,
        "MAE": float(np.mean(np.abs(err))),
        "RMSE": float(np.sqrt(np.mean(err ** 2))),
        "Pearson_r": r,
        "Bias": float(np.mean(err)),
        "LoA_Lower": float(np.mean(err) - 1.96 * np.std(err)),
        "LoA_Upper": float(np.mean(err) + 1.96 * np.std(err)),
    }

per_record = []
for rec, g in dfw.groupby("Record", sort=True):
    m = hr_validation_summary(g)
    m["Record"] = rec
    per_record.append(m)

global_metrics = hr_validation_summary(dfw)
hr_doc = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "dataset": "BIDMC",
    "fs": FS,
    "window_s": WINDOW_S,
    "step_s": STEP_S,
    "reference": "HR del monitor (no PULSE); sesgo = media(HR_est - HR_Ref); LoA = ±1.96·DE",
    "global": global_metrics,
    "per_record": per_record,
}
hr_bytes = save_json_local(hr_doc, "hr_validation_summary.json")

print("\nValidación de FC (global):")
for k, v in global_metrics.items():
    v_txt = "None" if v is None else (f"{v:.4f}" if isinstance(v, float) else f"{v}")
    print(f"  {k:10s}: {v_txt}")
print("\nValidación de FC por registro (primeras 10 filas):")
print(pd.DataFrame(per_record).set_index("Record").head(10).round(3).to_string())
print(f"\nhr_validation_summary.json → {hr_bytes:,} bytes")

In [ ]:
PLATA_DIR.mkdir(parents=True, exist_ok=True)

# 1. SpO2: colapso en normoxia
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(df["SpO2"], bins=np.arange(80, 102) - 0.5, color="steelblue", edgecolor="white")
ax.set_xlabel("SpO2 (%)")
ax.set_ylabel("Muestras")
ax.set_title("SpO2 — BIDMC (normoxia)")
fig.tight_layout()
fig.savefig(PLATA_DIR / "fig_spo2_hist.png", dpi=120)
plt.show()

# 2. Distribución de HR y RR
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(df["HR"], bins=40, color="indianred")
axes[0].set_xlabel("HR (bpm)")
axes[0].set_ylabel("Muestras")
axes[0].set_title("FC del monitor")
axes[1].hist(df["RR"], bins=40, color="seagreen")
axes[1].set_xlabel("RR (resp/min)")
axes[1].set_ylabel("Muestras")
axes[1].set_title("Frecuencia respiratoria")
fig.tight_layout()
fig.savefig(PLATA_DIR / "fig_hr_rr_hist.png", dpi=120)
plt.show()

# 3. Morfología media de un registro (pulsos alineados por pico sistólico)
rec0 = records_list[0]
clean0 = proc[rec0]["Clean"]
peaks0 = peaks_by_record[rec0]
pre, post = int(0.3 * FS), int(0.5 * FS)
cycles = np.array([clean0[p - pre:p + post] for p in peaks0 if p - pre >= 0 and p + post <= len(clean0)])
morph_mean = cycles.mean(axis=0)
morph_std = cycles.std(axis=0)
t_morph = np.arange(-pre, post) / FS
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(t_morph, morph_mean, color="black", label="Media")
ax.fill_between(t_morph, morph_mean - morph_std, morph_mean + morph_std, alpha=0.25, label="±1 DE")
ax.set_xlabel("Tiempo relativo al pico (s)")
ax.set_ylabel("PLETH filtrada (NU)")
ax.set_title(f"Morfología media — {rec0} (n={len(cycles)} pulsos)")
ax.legend()
fig.tight_layout()
fig.savefig(PLATA_DIR / "fig_morphology_mean.png", dpi=120)
plt.show()

# 4. SQI por ventana y Quality_Flag
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(dfw["SQI_templatematch"].dropna(), bins=40, color="slateblue")
axes[0].axvline(THR_SQI, color="red", linestyle="--", label=f"P10 = {THR_SQI:.3f}")
axes[0].set_xlabel("SQI templatematch")
axes[0].set_ylabel("Ventanas")
axes[0].set_title("SQI por ventana")
axes[0].legend()
flag_counts = dfw["Quality_Flag"].value_counts()
axes[1].bar(flag_counts.index, flag_counts.values, color=["seagreen", "indianred"][:len(flag_counts)])
axes[1].set_xlabel("Quality_Flag")
axes[1].set_ylabel("Ventanas")
axes[1].set_title("Calidad por ventana")
fig.tight_layout()
fig.savefig(PLATA_DIR / "fig_sqi_flags.png", dpi=120)
plt.show()

# 5. Perfusión AC/DC por ventana
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(dfw["AC_DC_pct"], bins=40, color="darkorange")
ax.axvline(THR_ACDC, color="red", linestyle="--", label=f"P10 = {THR_ACDC:.2f} %")
ax.set_xlabel("AC/DC (%)")
ax.set_ylabel("Ventanas")
ax.set_title("Perfusión AC/DC por ventana")
ax.legend()
fig.tight_layout()
fig.savefig(PLATA_DIR / "fig_acdc.png", dpi=120)
plt.show()

print("Figuras → plata/ppg/: fig_spo2_hist.png, fig_hr_rr_hist.png, fig_morphology_mean.png, fig_sqi_flags.png, fig_acdc.png")

In [ ]:
ORO_DIR.mkdir(parents=True, exist_ok=True)

frames = []
for rec in records_list:
    d = proc[rec]
    n = len(d["Clean"])
    frames.append(pd.DataFrame({
        "Dataset": "BIDMC",
        "Subject": d["Subject"],
        "Record": rec,
        "Sample_Index": np.arange(1, n + 1, dtype=np.int64),
        "PPG": d["Clean"].astype(np.float32),
        "PPG_Raw": d["Raw"].astype(np.float32),
        "SpO2_Ref": d["SpO2"].astype(np.float32),
        "HR_Ref": d["HR"].astype(np.float32),
        "RR_Ref": d["RR"].astype(np.float32),
    }))

df_oro = pd.concat(frames, ignore_index=True)

ORO_COLS = ["Dataset", "Subject", "Record", "Sample_Index", "PPG", "PPG_Raw", "SpO2_Ref", "HR_Ref", "RR_Ref"]
assert list(df_oro.columns) == ORO_COLS
assert df_oro[["PPG", "PPG_Raw", "SpO2_Ref", "HR_Ref", "RR_Ref"]].isna().sum().sum() == 0
assert len(df_oro) == EXPECTED_ROWS
assert df_oro["Record"].nunique() == EXPECTED_RECORDS
assert len(dfw) > 6000, f"Ventanas insuficientes: {len(dfw)}"
assert (df_oro.groupby("Record")["Sample_Index"].min() == 1).all()
assert (df_oro.groupby("Record")["Sample_Index"].max() == 60001).all()

oro_path = ORO_DIR / "set_a.parquet"
df_oro.to_parquet(oro_path, index=False)

csv_bytes = save_csv_local(dfw, "ppg_quality_per_window_125hz.csv")

print(f"Export: {len(df_oro):,} filas | {df_oro['Record'].nunique()} records | 0 NaN en PPG/PPG_Raw/SpO2_Ref/HR_Ref/RR_Ref")
print(f"set_a.parquet → {oro_path.stat().st_size:,} bytes")
print(f"ppg_quality_per_window_125hz.csv → {csv_bytes:,} bytes ({len(dfw):,} ventanas)")

## Cierre

**Pipeline C1 completado:** `BIDMC-Reduced.csv` → filtrado Elgendi (NeuroKit2) → ventanas de 8 s con paso 4 s → SQI por ventana (templatematch + SNR + skewness/kurtosis + AC/DC) y `Quality_Flag` (flag duro ⇒ `low-quality`; ≥2 fallas por percentil ⇒ `low-quality`) → validación de FC contra `HR` del monitor → `set_a.parquet` en `oro/ppg`.

### Artefactos generados
* `plata/ppg/bidmc_audit.json` — auditoría física por registro.
* `plata/ppg/ppg_quality_config.json` — umbrales P10 y conteos de `Quality_Flag`.
* `plata/ppg/hr_validation_summary.json` — MAE/RMSE/Pearson/Bland-Altman global y por registro.
* `plata/ppg/ppg_quality_per_window_125hz.csv` — una fila por ventana.
* `plata/ppg/fig_*.png` — EDA (SpO2, HR/RR, morfología, SQI/flags, AC/DC).
* `oro/ppg/set_a.parquet` — BIDMC a 125 Hz con el esquema `Dataset, Subject, Record, Sample_Index, PPG, PPG_Raw, SpO2_Ref, HR_Ref, RR_Ref`.

### Limitaciones
* **Normoxia:** SpO2 83–100 %; no hay desaturaciones para validar SpO2.
* **Monocanal:** `PLETH` procesado por el monitor, sin canales Rojo/IR.
* **Sin ACC:** no se valida SQI de movimiento (registros de UCI inmóvil).
* **pyPPG bloqueado** en este stack (NumPy 2); el reintento queda para C2.

### Estado
* **C2 bloqueado** hasta contar con canales Rojo/IR y ground truth independiente de SpO2 (ver plan).